In [ ]:
from qick import *
# %matplotlib widget
%matplotlib notebook
# %matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator

import xarray as xr

In [ ]:
import os
import sys
sys.path.insert(0, '../../pattern/')
sys.path.insert(0, '../../instrument/')

In [ ]:
from pathlib import Path

folder_name = Path.cwd().name
data_dir = Path("Z:/labdata/qcdlabs") / folder_name
data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# from Valon.instr_Valon5015 import Valon5015
# from Agilent.instr_N9928A import N9928A
from RS.instr_FSV40 import FSV40

spec_address = "10.0.100.226"
spec = FSV40(spec_address)

In [ ]:
sys.path.insert(0, '../../pattern/')
from helper_sweep import do_sweep

from double_conversion_mixer.instr_double_conversion_mixer import DuoMixer

lo1_address = "10.0.100.24"
lo2_address = ["10.0.100.32"]
drive = DuoMixer(lo1_address, lo2_address)

In [ ]:
from xilinx_qick.class_drx import drx
from xilinx_qick.class_rox import rox
from xilinx_qick.class_sweep import sweep
from xilinx_qick.instr_xilinx_v1 import XilinxProg

xilinx_1 = XilinxProg(ip_address="10.0.100.21", mode='AveragerProgram')

In [ ]:
xilinx_1.reps = int(1e9)
dr_ch0 = 0
ro_ch0 = 0

In [ ]:
dt = 10/9830.4
# dt = 1e-3
t_gen = np.arange(0, 1.6*2, dt)

if_freq1 = 5e6
if_freq2 = -2.5e6

# s_data = 0.5 + 0.5 * np.linspace(0,1,len(t_data))

s_gen = 1 * np.exp(-1j*2*np.pi*(0/1e6) * t_gen)
s_gen += 0.5 * np.exp(-1j*2*np.pi*(if_freq1/1e6) * t_gen)
s_gen += 0.25 * np.exp(-1j*2*np.pi*(if_freq2/1e6) * t_gen)

# plt.figure()
# plt.scatter(t_data, s_data.real, marker='.')
# plt.scatter(t_data, s_data.imag, marker='.')
# plt.ylim(-3, 3)
# plt.show()

fig = plt.figure(figsize=(6, 6))
gs = fig.add_gridspec(2, 1)
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])

ax0.scatter(t_gen, np.abs(s_gen), marker='.')
ax1.scatter(t_gen, np.angle(s_gen), marker='.')
# plt.ylim(-3, 3)
plt.show()

In [ ]:
if_frequency = 5.5e9
dr_readout = drx(soc=xilinx_1.soccfg,
                 dr_ch=dr_ch0, ro_ch=ro_ch0,
                 frequency= if_frequency / 1e6, gain=1, phase=0, nqz=2)
ph_0 = 1

dr_readout.wave.add(name='x2', t_data=t_gen, s_data=s_gen/ph_0, idx=-1, interp_order=0)
dr_readout.rox.set(frequency=if_frequency / 1e6, length=0.01, delay=0.0, sleep=0.0)
xilinx_1.add(dr_readout=dr_readout)

In [ ]:
lo1_frequency = 8.45e9
set_frequency_hz = 8.1e9

drive.set_lo1(frequency=lo1_frequency, power=17)
drive.set_lo2(idx=0, power=17)
drive.set_frequency(idx=0, set_frequency_hz=set_frequency_hz, if_frequency_hz = if_frequency)

drive_freq_list = np.arange(0.3, 8.3+1e-9, 0.5) * 1e9

In [ ]:
# f0 = set_frequency_hz - 0.5e8
# f1 = set_frequency_hz + 0.5e8
f0 = if_frequency - 0.5e8
f1 = if_frequency + 0.5e8

spec.start_frequency(f0)
spec.stop_frequency(f1)
spec.if_frequency(1e5)
spec.video_frequency(1e5)
spec.average(10)
spec.points(1001)

In [ ]:
span_frequency = 100e6
mirror_frequancy = 9830.4e6/2

center_freq_list = np.array([if_frequency, 2*mirror_frequancy-if_frequency])
# center_freq_list = np.array([2, 4, 9.7, 11.7]) * 1e9

start_freq_list = center_freq_list - span_frequency/2
stop_freq_list = center_freq_list + span_frequency/2

In [ ]:
config_device = [spec]
config_sweep = [
    [
        [spec.start_frequency, 'start_frequency_hz', start_freq_list, False],
        [spec.stop_frequency, 'stop_frequency_hz', stop_freq_list, False]
    ],
]

In [ ]:
# drive.lo1.output(1)
# drive.lo2[0].output(1)
file_name= data_dir / 'test_1.zarr'

xilinx_1.test(load_pulses=True, progress=False)
do_sweep(config_device, config_sweep, spec.get_trace, file_name)

xilinx_1.soc.reset_gens()
drive.lo1.output(0)
drive.lo2[0].output(0)

In [ ]:
file_name= data_dir / 'test_1.zarr'
with xr.open_zarr(file_name, consolidated=False) as f:
    raw_data = f['spectrum']

plot_frequency_ghz = raw_data.FSV40_frequency_hz.data/1e9
_diff = np.diff(plot_frequency_ghz)
idx_list = np.where(np.abs(_diff) > 5*np.abs(plot_frequency_ghz[1]-plot_frequency_ghz[0]))[0]

fig = plt.figure(figsize=(10, 5))
gs = fig.add_gridspec(1, len(idx_list)+1)

val_idx_pre = 0
for num_idx, val_idx in enumerate(idx_list):
    ax = fig.add_subplot(gs[num_idx])

    ax.plot(plot_frequency_ghz[val_idx_pre:val_idx+1], np.real(raw_data.isel(FSV40_frequency_hz=slice(val_idx_pre,val_idx+1)).squeeze().data.compute()))
    val_idx_pre = val_idx+1
    ax.set_ylim(-95, -20)

ax = fig.add_subplot(gs[num_idx+1])
ax.plot(plot_frequency_ghz[val_idx_pre:], np.real(raw_data.isel(FSV40_frequency_hz=slice(val_idx_pre,None)).squeeze().data.compute()))
ax.set_ylim(-95, -20)

plt.savefig('test_1.pdf')
plt.show()